In [ ]:
# ====================== CELL 1: Imports & Global Seeds (REPRODUCIBLE) ======================
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt
from gymnasium.wrappers import RecordVideo
import os

# ====================== GLOBAL SEED FOR REPRODUCIBILITY ======================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} | Seed set to {SEED} (fully reproducible)")

In [ ]:
class ReplayBuffer:
    def __init__(self, action_size, buffer_size, batch_size, seed):
        self.action_size = action_size
        self.memory = deque(maxlen=buffer_size)
        self.batch_size = batch_size
        random.seed(seed)

    def add(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def sample(self):
        experiences = random.sample(self.memory, k=self.batch_size)
        states = torch.from_numpy(np.vstack([e[0] for e in experiences])).float().to(device)
        actions = torch.from_numpy(np.vstack([e[1] for e in experiences])).long().to(device)
        rewards = torch.from_numpy(np.vstack([e[2] for e in experiences])).float().to(device)
        next_states = torch.from_numpy(np.vstack([e[3] for e in experiences])).float().to(device)
        dones = torch.from_numpy(np.vstack([e[4] for e in experiences]).astype(np.uint8)).float().to(device)
        return (states, actions, rewards, next_states, dones)

    def __len__(self):
        return len(self.memory)

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_size, action_size, seed=42):
        super(QNetwork, self).__init__()
        self.seed = torch.manual_seed(seed)
        # You can experiment with depth here if you want
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
class DQNAgent:
    def __init__(self, state_size, action_size, seed,
                 lr=5e-4, buffer_size=int(1e5), batch_size=64,
                 gamma=0.99, tau=1e-3, update_every=4):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = gamma
        self.tau = tau
        self.update_every = update_every
        self.batch_size = batch_size

        self.qnetwork_local = QNetwork(state_size, action_size, seed).to(device)
        self.qnetwork_target = QNetwork(state_size, action_size, seed).to(device)
        self.optimizer = optim.Adam(self.qnetwork_local.parameters(), lr=lr)

        self.memory = ReplayBuffer(action_size, buffer_size, batch_size, seed)
        self.t_step = 0
        self.loss_history = []

    def step(self, state, action, reward, next_state, done):
        self.memory.add(state, action, reward, next_state, done)
        self.t_step = (self.t_step + 1) % self.update_every
        if self.t_step == 0 and len(self.memory) > self.batch_size:
            experiences = self.memory.sample()
            loss = self.learn(experiences)
            if loss is not None:
                self.loss_history.append(loss)

    def act(self, state, eps=0.0):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        self.qnetwork_local.eval()
        with torch.no_grad():
            action_values = self.qnetwork_local(state)
        self.qnetwork_local.train()
        if random.random() > eps:
            return np.argmax(action_values.cpu().data.numpy())
        else:
            return random.choice(np.arange(self.action_size))

    def learn(self, experiences):
        states, actions, rewards, next_states, dones = experiences

        Q_targets_next = self.qnetwork_target(next_states).detach().max(1)[0].unsqueeze(1)
        Q_targets = rewards + (self.gamma * Q_targets_next * (1 - dones))

        Q_expected = self.qnetwork_local(states).gather(1, actions)

        loss = F.mse_loss(Q_expected, Q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Soft update target network
        for target_param, local_param in zip(self.qnetwork_target.parameters(), self.qnetwork_local.parameters()):
            target_param.data.copy_(self.tau * local_param.data + (1.0 - self.tau) * target_param.data)

        return loss.item()

In [ ]:
# ====================== CELL 5: Training (1000 episodes) - REPRODUCIBLE ======================
env = gym.make('LunarLander-v3')

state_size = env.observation_space.shape[0]
action_size = env.action_space.n
print(f"State size: {state_size}, Action size: {action_size}")

agent = DQNAgent(state_size=state_size, action_size=action_size, seed=SEED)

n_episodes = 1000
max_t = 1000
eps_start = 1.0
eps_end = 0.01
eps_decay = 0.995
scores = []
eps = eps_start

print("=== Starting training ===")
for i_episode in range(1, n_episodes + 1):
    # Seed environment for reproducibility (same initial conditions every run)
    state, _ = env.reset(seed=SEED + i_episode)
    score = 0
    for t in range(max_t):
        action = agent.act(state, eps)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        agent.step(state, action, reward, next_state, done)
        state = next_state
        score += reward
        if done:
            break
    scores.append(score)
    eps = max(eps_end, eps_decay * eps)

    if i_episode % 50 == 0:
        avg_score = np.mean(scores[-100:])
        print(f"Episode {i_episode}\tAverage Score (last 100): {avg_score:.2f}\tEpsilon: {eps:.3f}")

env.close()

In [ ]:
# Reward plot
plt.figure(figsize=(12, 5))
plt.plot(scores, label='Episode Reward')
plt.title('Learning Progress - Rewards per Episode (0-1000)')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.legend()
plt.grid()
plt.savefig('rewards_plot.png', dpi=200)
plt.show()

# Smoothed reward
window = 50
if len(scores) >= window:
    smoothed = np.convolve(scores, np.ones(window)/window, mode='valid')
    plt.figure(figsize=(12, 5))
    plt.plot(smoothed, label='Smoothed Reward')
    plt.title('Smoothed Rewards (window=50)')
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.legend()
    plt.grid()
    plt.savefig('smoothed_rewards.png', dpi=200)
    plt.show()

# Loss plot
if len(agent.loss_history) > 0:
    plt.figure(figsize=(12, 5))
    plt.plot(agent.loss_history)
    plt.title('Training Loss')
    plt.xlabel('Update Step')
    plt.ylabel('MSE Loss')
    plt.grid()
    plt.savefig('loss_plot.png', dpi=200)
    plt.show()

print("All plots saved!")

In [ ]:
def record_agent_video(agent, video_folder="./videos", n_episodes=3):
    os.makedirs(video_folder, exist_ok=True)
    env = gym.make('LunarLander-v3', render_mode="rgb_array")
    env = RecordVideo(env, video_folder=video_folder, episode_trigger=lambda x: True)
    for i in range(n_episodes):
        state, _ = env.reset()
        done = False
        while not done:
            action = agent.act(state, eps=0.0)
            state, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
    env.close()
    print(f"Videos saved in {video_folder}/ folder")

record_agent_video(agent)